In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout

In [2]:
# List of file paths to your CSV files
file_paths = ['E:\\project_work\\dehradun 2015-01-01 to 2015-12-31.csv', 
              'E:\\project_work\\dehradun 2016-01-01 to 2017-12-31.csv', 
              'E:\\project_work\\dehradun 2018-01-01 to 2019-12-31.csv', 
              'E:\\project_work\\dehradun 2020-01-01 to 2021-12-31.csv', 
              'E:\\project_work\\dehradun 2022-01-01 to 2023-12-31.csv']
# Initialize an empty list to store the dataframes
dfs = []

# Loop through each file and read it into a DataFrame
for file in file_paths:
    df = pd.read_csv(file)
    dfs.append(df)

# Concatenate all the DataFrames into a single DataFrame
full_data = pd.concat(dfs, ignore_index=True)

# Convert 'datetime' column to datetime type if it's not already
full_data['datetime'] = pd.to_datetime(full_data['datetime'])

# Sort the data by the 'datetime' column
full_data = full_data.sort_values(by='datetime')

# save the csv
# full_data.to_csv('edited_weather_data.csv', index=False)

full_data = full_data.drop(['name', 'feelslikemax', "feelslikemin", "precipcover", "preciptype", "severerisk", "moonphase", "conditions", "description", "icon","stations", "snow", "snowdepth", 'uvindex','precip', 'windspeed','precipprob','cloudcover','visibility', 'winddir'], axis=1)



In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Assuming you have loaded your full_data DataFrame
# removed windgust, solar..,sealevelpressure
# 1. Display descriptive statistics
print(full_data.describe())

# 2. Convert 'datetime' to datetime object
full_data['datetime'] = pd.to_datetime(full_data['datetime'])

# 3. Check for missing values
print(full_data.isnull().sum())

# 4. Fill missing values or drop rows/columns as needed
# full_data = full_data.ffill()  # Forward filling

# Ensure any missing values in visibility are handled
# full_data = full_data.assign(visibility=full_data['visibility'].fillna(full_data['visibility'].mean()))

# 5. Encode 'precipprob' to binary
# full_data['precipprob'] = full_data['precipprob'].apply(lambda x: 1 if x == 100 else 0)

# 6. Normalize 'temp' using MinMaxScaler
temp_scaler = MinMaxScaler()
full_data[['temp']] = temp_scaler.fit_transform(full_data[['temp']])

# 7. Normalize other features using MinMaxScaler
# scaler = MinMaxScaler()
# full_data[['temp', 'tempmax', 'tempmin', 'feelslike', 'dew', 'humidity',
#            'precip', 'precipprob', 'windspeed',
#            'cloudcover', 'visibility', 'windgust', 'solarradiation', 'solarenergy', 'uvindex', 'sealevelpressure']] = scaler.fit_transform(
#                full_data[['temp', 'tempmax', 'tempmin', 'feelslike', 'dew', 'humidity',
#                           'precip', 'precipprob', 'windspeed', 'windgust', 'solarradiation', 'solarenergy', 'uvindex', 'sealevelpressure',
#                           'cloudcover', 'visibility']]
#            )
scaler = MinMaxScaler()
full_data[['temp', 'tempmax', 'tempmin', 'feelslike', 'dew', 'humidity',
            'windgust', 'solarradiation', 'solarenergy', 'sealevelpressure']] = scaler.fit_transform(
               full_data[['temp', 'tempmax', 'tempmin', 'feelslike', 'dew', 'humidity',
                            'windgust', 'solarradiation', 'solarenergy', 'sealevelpressure',
                           ]]
           )
# 8. Normalize wind direction
# full_data['winddir_rad'] = np.deg2rad(full_data['winddir'])  # Convert to radians
# full_data['winddir_sin'] = np.sin(full_data['winddir_rad'])  # Sine transformation
# full_data['winddir_cos'] = np.cos(full_data['winddir_rad'])  # Cosine transformation

# 9. Preprocess sunrise and sunset times
# Convert 'sunrise' and 'sunset' to datetime objects
full_data['sunrise'] = pd.to_datetime(full_data['sunrise'])
full_data['sunset'] = pd.to_datetime(full_data['sunset'])

# Calculate minutes since midnight for sunrise and sunset
full_data['sunrise_minutes'] = full_data['sunrise'].dt.hour * 60 + full_data['sunrise'].dt.minute
full_data['sunset_minutes'] = full_data['sunset'].dt.hour * 60 + full_data['sunset'].dt.minute

# Normalize sunrise and sunset minutes
full_data[['sunrise_minutes', 'sunset_minutes']] = scaler.fit_transform(
    full_data[['sunrise_minutes', 'sunset_minutes']]
)

# 10. Drop unnecessary columns
full_data.drop(['datetime', 'sunrise', 'sunset'], axis=1, inplace=True)

# 11. Display descriptive statistics for the normalized data
print(full_data[['temp', 'tempmax', 'tempmin', 'sunrise_minutes', 'sunset_minutes']].describe())


                            datetime      tempmax      tempmin         temp  \
count                           3287  3287.000000  3287.000000  3287.000000   
mean   2019-07-01 23:59:59.999999744    28.852297    16.472224    21.851202   
min              2015-01-01 00:00:00    11.400000     1.100000     7.800000   
25%              2017-04-01 12:00:00    24.950000    10.700000    16.500000   
50%              2019-07-02 00:00:00    29.800000    17.400000    23.700000   
75%              2021-09-30 12:00:00    32.600000    22.700000    26.900000   
max              2023-12-31 00:00:00    43.600000    27.700000    35.300000   
std                              NaN     5.600972     6.551562     6.180714   

         feelslike          dew     humidity     windgust  sealevelpressure  \
count  3287.000000  3287.000000  3287.000000  3287.000000       3287.000000   
mean     22.413264    14.627502    68.415972    24.023213       1008.922939   
min       7.800000    -3.600000    18.700000     4.

In [4]:
file_path = "processed_data.csv"
full_data.to_csv(file_path, index=False)

In [5]:
import numpy as np

# Function to create sequences
def create_sequences(data, target_column, time_steps):
    sequences = []
    targets = []

    for i in range(len(data) - time_steps):
        seq = data.iloc[i:i + time_steps].values  # Create sequence
        target = data.iloc[i + time_steps][target_column]  # Target value
        sequences.append(seq)
        targets.append(target)

    return np.array(sequences), np.array(targets)

# Define your target variable and time steps
target_column = 'temp'  # Change this if you want to predict a different variable
time_steps = 8  # Adjust this according to your needs

# Create sequences
X, y = create_sequences(full_data, target_column, time_steps)

# Print the shape of the sequences
print("Shape of X:", X.shape)  # (number_of_samples, time_steps, number_of_features)
print("Shape of y:", y.shape)  # (number_of_samples,)


Shape of X: (3279, 8, 12)
Shape of y: (3279,)


In [6]:

from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# # Further split training into training and validation sets
# X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)
# Time-based split (assuming data is sorted by time)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, shuffle=False)  # 80% for training
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, shuffle=False)  # 50% of the remaining for validation and test

print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)


Training set shape: (2623, 8, 12) (2623,)
Validation set shape: (328, 8, 12) (328,)
Test set shape: (328, 8, 12) (328,)


In [7]:
import GPyOpt
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, BatchNormalization, LSTM, Dropout, Dense
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping

# Define the hyperparameter bounds for optimization
bounds = [
    {'name': 'learning_rate', 'type': 'continuous', 'domain': (1e-6, 1e-2)},  # Learning rate
    {'name': 'batch_size', 'type': 'discrete', 'domain': (8, 16, 32, 64)},  # Batch size
    {'name': 'conv1_filters', 'type': 'discrete', 'domain': (64, 82, 128)},  # Filters for Conv1D
    {'name': 'kernel_size', 'type': 'discrete', 'domain': (3, 5, 6)},  # Kernel size for Conv1D
    {'name': 'pool_size', 'type': 'discrete', 'domain': (2, 3)},  # Pool size for MaxPooling1D
    {'name': 'lstm_units1', 'type': 'discrete', 'domain': (100, 200, 300)},  # LSTM units (first layer)
    {'name': 'lstm_units2', 'type': 'discrete', 'domain': (200, 300, 400)},  # LSTM units (second layer)
    {'name': 'dropout_rate1', 'type': 'continuous', 'domain': (0.1, 0.5)},  # Dropout rate (first)
    {'name': 'dropout_rate2', 'type': 'continuous', 'domain': (0.1, 0.5)},  # Dropout rate (second)
]

In [8]:
# Global counter to track iterations
iteration_counter = 0

def objective_function(params):
    global iteration_counter
    iteration_counter += 1  # Increment iteration counter
    print(f"Iteration number: {iteration_counter}")  # Print the current iteration
    
    # Unpack hyperparameters from the optimization process
    learning_rate, batch_size, conv1_filters, kernel_size, pool_size, lstm_units1, lstm_units2, dropout_rate1, dropout_rate2 = params[0]
    
    # Cast the discrete parameters to integers
    conv1_filters = int(conv1_filters)
    kernel_size = int(kernel_size)
    pool_size = int(pool_size)
    lstm_units1 = int(lstm_units1)
    lstm_units2 = int(lstm_units2)
    batch_size = int(batch_size)
    
    # Define the model architecture with the current hyperparameters
    model = Sequential()
    # Update the Conv1D input_shape to match the data
    model.add(Conv1D(filters=conv1_filters, kernel_size=kernel_size, activation='relu', input_shape=(8, 12)))
    model.add(MaxPooling1D(pool_size=pool_size))
    
    model.add(LSTM(lstm_units1, return_sequences=True))
    model.add(Dropout(dropout_rate1))
    model.add(LSTM(lstm_units2))
    model.add(Dropout(dropout_rate2))
    
    model.add(Dense(1))  # Regression output layer
    
    # Compile the model with the current learning rate
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mean_squared_error', metrics=['mae'])
    
    # Add early stopping
    early_stopping = EarlyStopping(
        monitor='val_loss',  # Monitor validation loss
        patience=40,         # Stop training after 40 epochs with no improvement
        restore_best_weights=True  # Restore weights from the epoch with the best validation loss
    )
    
    # Train the model with the given batch size and hyperparameters
    history = model.fit(
        X_train, y_train, 
        epochs=200, 
        batch_size=batch_size, 
        validation_data=(X_val, y_val), 
        verbose=0,
        callbacks=[early_stopping]  # Include early stopping in the callbacks
    )
    # Get the validation loss (this is the value to minimize)
    val_loss = history.history['val_loss'][-1]
    return val_loss

# Set up the optimizer for Bayesian Optimization
optimizer = GPyOpt.methods.BayesianOptimization(
    f=objective_function,  # Objective function to minimize
    domain=bounds,         # Hyperparameter search space
    acquisition_type='EI',  # Acquisition function (Expected Improvement)
    acquisition_jitter=0.05 # Jitter to promote exploration
)
# Run the optimization process
optimizer.run_optimization(max_iter=300)  # Run for a specified number of iterations

# Display the optimal hyperparameters
print("Optimal hyperparameters:", optimizer.x_opt)

Iteration number: 1
Iteration number: 2
Iteration number: 3
Iteration number: 4
Iteration number: 5
Iteration number: 6
Iteration number: 7
Iteration number: 8
Iteration number: 9
Iteration number: 10
Iteration number: 11
Iteration number: 12
Iteration number: 13
Iteration number: 14
Iteration number: 15
Iteration number: 16
Iteration number: 17
Iteration number: 18
Iteration number: 19
Iteration number: 20
Iteration number: 21
Iteration number: 22
Iteration number: 23
Iteration number: 24
Iteration number: 25
Iteration number: 26
Iteration number: 27
Iteration number: 28
Iteration number: 29
Iteration number: 30
Iteration number: 31
Iteration number: 32
Iteration number: 33
Iteration number: 34
Iteration number: 35
Iteration number: 36
Iteration number: 37
Iteration number: 38
Iteration number: 39
Iteration number: 40
Iteration number: 41
Iteration number: 42
Iteration number: 43
Iteration number: 44
Iteration number: 45
Iteration number: 46
Iteration number: 47
Iteration number: 48
I

In [9]:
# Print the results of the optimization
print("Best hyperparameters:")
param_names = [b['name'] for b in bounds]
best_params = optimizer.x_opt
for name, value in zip(param_names, best_params):
    print(f"{name}: {value}")
print("Best validation loss:", optimizer.fx_opt)

Best hyperparameters:
learning_rate: 0.0053739015411619325
batch_size: 64.0
conv1_filters: 82.0
kernel_size: 3.0
pool_size: 2.0
lstm_units1: 300.0
lstm_units2: 400.0
dropout_rate1: 0.164235338780274
dropout_rate2: 0.34377197804621473
Best validation loss: 0.0017037037760019302
